In [14]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym
import mani_skill.envs
from mani_skill.agents.controllers import PDEEPoseControllerConfig

env = gym.make(
    "PickCube-v1", # there are more tasks e.g. "PushCube-v1", "PegInsertionSide-v1", ...
    num_envs=4,
    obs_mode="state", # there is also "state_dict", "rgbd", ...
    control_mode="pd_joint_delta_pos", 
    render_mode="rgb_array",
    robot_uids="so100"
)
# print("Observation space", env.observation_space)
print("Observation space", env.observation_space)
print("Action space", env.action_space)

Observation space Box(-inf, inf, (4, 36), float32)
Action space Box(-1.0, 1.0, (4, 6), float32)


In [15]:
import numpy as np

env.reset()
episode_over = False
frames = [env.render().cpu().numpy()]
while not episode_over:
    action = env.action_space.sample()
    observation, reward, terminated, truncated, info = env.step(action)
    frames.append(env.render().cpu().numpy())
    episode_over = all(terminated | truncated)
frames = np.stack(frames, axis=0)

In [16]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display

fig = plt.figure(num=1, clear=True, figsize=(4.5, 5))
axes = []
for i in range(frames.shape[1]):
    axes.append(fig.add_subplot(2, 2, i+1))
    axes[i].set(xticks=[], xticklabels=[], yticks=[], yticklabels=[])
    axes[i].imshow(frames[0][i])
fig.tight_layout()

ims = []
for i in range(frames.shape[0]):
    images = []
    for j in range(frames.shape[1]):
        image = axes[j].imshow(frames[i][j], animated=True)
        images.append(image)
    ims.append(images)

ani = animation.ArtistAnimation(fig, ims, interval=100, blit=True, repeat_delay=1000)
plt.close(fig=1)
display(HTML(ani.to_html5_video()))